# Classification: Temporal Features

Instead of averaging metrics across all trials, we split each session into the first third (early) and last third (late) of session. For each segment we compute the mean of each metric separately, then add delta features (late minus early) to capture within-session change. This picks up attention deterioration, fatigue, and habituation patterns that full-session averages wash out.

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

from src.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Build temporal features (early / late / delta)

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

BASE_METRICS = [
    "fixation_count",
    "mean_fixation_duration_ms",
    "total_fixation_duration_ms",
    "fixation_rate_per_sec",
    "fixation_bias",
    "scanpath_length",
    "saccade_count",
    "saccade_rate_per_sec",
    "mean_saccade_amplitude",
    "blink_count",
    "blink_rate_per_min",
    "transition_matrix_density",
    "gaze_transition_entropy",
    "first_fixation_duration_ms",
    "second_fixation_duration_ms",
    "dwell_time_ms_negative",
    "dwell_time_ms_positive",
    "dwell_time_ms_neutral",
    "dwell_time_500ms_negative",
    "dwell_time_500ms_positive",
    "dwell_time_500ms_neutral",
    "fixation_proportion_negative",
    "fixation_proportion_positive",
    "fixation_proportion_neutral",
    "fixation_count_negative",
    "fixation_count_positive",
    "fixation_count_neutral",
    "revisit_count_negative",
    "revisit_count_positive",
    "revisit_count_neutral",
    "ttff_ms_negative",
    "ttff_ms_positive",
    "ttff_ms_neutral",
]

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))
numbered = numbered.withColumn("n_trials", F.count("*").over(Window.partitionBy("session_id")))

early = numbered.filter(F.col("trial_num") <= F.col("n_trials") / 3)
late = numbered.filter(F.col("trial_num") > F.col("n_trials") * 2 / 3)

def build_agg(suffix):
    agg_exprs = []
    for m in BASE_METRICS:
        agg_exprs.append(F.mean(m).alias(f"{m}_{suffix}"))
    for valence in ["negative", "positive", "neutral"]:
        agg_exprs.append(
            F.avg(F.when(F.col("first_fixation_valence") == valence, 1).otherwise(0))
             .alias(f"first_fix_prob_{valence}_{suffix}"))
        agg_exprs.append(
            F.avg(F.when(F.col("second_fixation_valence") == valence, 1).otherwise(0))
             .alias(f"second_fix_prob_{valence}_{suffix}"))
    return agg_exprs

early_metrics = early.groupBy("session_id").agg(*build_agg("early"))
late_metrics = late.groupBy("session_id").agg(*build_agg("late"))

session_metrics = early_metrics.join(late_metrics, on="session_id", how="inner")

df_joined = session_metrics.join(
    df_forms.select("session_id", "uid", "phq9_score", "phq9_severity"),
    on="session_id", how="inner",
)

df = df_joined.toPandas()

for m in BASE_METRICS:
    df[f"{m}_delta"] = df[f"{m}_late"] - df[f"{m}_early"]
for valence in ["negative", "positive", "neutral"]:
    df[f"first_fix_prob_{valence}_delta"] = df[f"first_fix_prob_{valence}_late"] - df[f"first_fix_prob_{valence}_early"]
    df[f"second_fix_prob_{valence}_delta"] = df[f"second_fix_prob_{valence}_late"] - df[f"second_fix_prob_{valence}_early"]

print(f"Sessions: {len(df)}, Users: {df['uid'].nunique()}")

## 2. Define targets and feature sets

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)
df["phq9_severity_class"] = df["phq9_severity"].map(
    {"minimal": 0, "mild": 1, "moderate": 2, "moderately_severe": 3, "severe": 4}
)

print(f"Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"Multi-class: {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

In [0]:
EARLY_FEATURES = [f"{m}_early" for m in BASE_METRICS]
LATE_FEATURES = [f"{m}_late" for m in BASE_METRICS]
DELTA_FEATURES = [f"{m}_delta" for m in BASE_METRICS]

PROB_EARLY = [f"{p}_{v}_early" for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]
PROB_LATE = [f"{p}_{v}_late" for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]
PROB_DELTA = [f"{p}_{v}_delta" for p in ["first_fix_prob", "second_fix_prob"] for v in ["negative", "positive", "neutral"]]

ALL_FEATURES = EARLY_FEATURES + LATE_FEATURES + DELTA_FEATURES + PROB_EARLY + PROB_LATE + PROB_DELTA

THEORY_BASE = [
    "fixation_bias", "dwell_time_ms_negative", "dwell_time_ms_positive",
    "fixation_proportion_negative", "fixation_proportion_positive",
    "revisit_count_negative", "revisit_count_positive",
    "blink_rate_per_min", "scanpath_length",
]
THEORY_FEATURES = []
for m in THEORY_BASE:
    THEORY_FEATURES.extend([f"{m}_early", f"{m}_late", f"{m}_delta"])
THEORY_FEATURES.extend([f"first_fix_prob_{v}_{s}" for v in ["negative", "positive"] for s in ["early", "late", "delta"]])

DELTA_ONLY_FEATURES = DELTA_FEATURES + PROB_DELTA

FEATURE_SETS = {
    "All Temporal": ALL_FEATURES,
    "Theory-Driven Temporal": THEORY_FEATURES,
    "Delta Only": DELTA_ONLY_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = ["phq9_score", "phq9_depressed", "phq9_severity_class"]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification (PHQ-9 >= 10)

### 4.1 Run classification

In [0]:
y_binary = df_clean["phq9_depressed"].values
binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_binary, groups)

### 4.2 Results

In [0]:
print(binary_df.to_string(index=False))

### 4.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_binary, groups, binary_df)

## 5. Multi-class classification (5 severity groups)

### 5.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_multi = df_clean["phq9_severity_class"].values.astype(int)
multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups)

### 5.2 Results

In [0]:
print(multi_df.to_string(index=False))

### 5.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_multi, groups, multi_df, PHQ9_LABELS)

---

## 6. Regression (predict PHQ-9 score)

### 6.1 Run regression

In [0]:
y_reg = df_clean["phq9_score"].values
reg_df = run_regression(df_clean, FEATURE_SETS, y_reg, groups)

### 6.2 Results

In [0]:
print(reg_df.to_string(index=False))

### 6.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_reg, groups, reg_df)

## 7. Feature Importance

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_binary, title="Feature importance (binary, all temporal)")

## 8. Summary

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(binary_df, multi_df, reg_df, feature_order, title="Temporal features")